In [7]:
# @title
%%bash
pip install gdown &
gdown 1XMItFk-UZUdt-zBkys493v2huZLL_wjl --output fact_ads.csv &
gdown 1VCChoxpkNi3FRKD8mUnDLHm6Rk7F5VP7 --output fact_payment.csv &
gdown 1I9kNMf_cujvt1T4KL7gsHaGV8zKHYm55 --output fact_reg.csv &
curl https://clickhouse.com/ | sh


Will download https://builds.clickhouse.com/master/amd64/clickhouse into clickhouse.2


Successfully downloaded the ClickHouse binary, you can run it as:
    ./clickhouse.2

You can also install it:
sudo ./clickhouse.2 install


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2911    0  2911    0     0   5076      0 --:--:-- --:--:-- --:--:--  5080
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  129M  100  129M    0     0  44.4M      0  0:00:02  0:00:02 --:--:-- 44.4M
Downloading...
From: https://drive.google.com/uc?id=1VCChoxpkNi3FRKD8mUnDLHm6Rk7F5VP7
To: /content/fact_payment.csv
100%|██████████| 92.6k/92.6k [00:00<00:00, 25.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1I9kNMf_cujvt1T4KL7gsHaGV8zKHYm55
To: /content/fact_reg.csv
100%|██████████| 88.8k/88.8k [00:00<00:00, 18.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1XMItFk-UZUdt-zBkys493v2huZLL_wjl
To: /content/fact_ads.csv
100%|██████████| 44.6k/44.6k [00:00<00:00, 60.5MB/s]


# Description
(If familiar, jump to Solution.)

## Formulation

Pre-requirements:

- Make a copy of this workbook to your own Google Drive

---

Based on the given data, create visualizations for advertisements using the following metrics:

- **installs** (number of registrations)
- **revenue** (sum of payments)
- **ROMI**

Feel free to include additional metrics if you consider them relevant, and explain why they were included.

**Note:** It is assumed that the ad impression day matches the user registration day.
In the workbook, the queries run via a ClickHouse client. Please use the appropriate SQL dialect in your queries.

## Task Requirements

The task must be completed using SQL only, meaning all data preparation (cleaning, transformation, etc.) should be done by **single SQL query** before using the prepared data mart in the visualization tool (use it only for aggregation or any complex calculations you need). For visualization, it is preferable to use any available BI tool. If none are available, you can use MS Excel / Google Sheets or any Python plotting library.

As a result of the task, you should provide:

- A Colab workbook containing the SQL queries and the resulting CSV output
- Screenshots or finalized visualizations of the reports

# Solution

## Result files

### Main

- [Ivan_Orlov_test_task.ipynb](https://colab.research.google.com/drive/1AzSY4DJTS7Tu1fBkguqmqNS_Z_LnCPPZ?usp=sharing) – (this very) collab containing the SQL query and the resulting CSV output.
- [Ivan_Orlov_Viz.twbx](https://public.tableau.com/app/profile/ivan.orlov/viz/Ivan_Orlov_Viz/Main) – finalized visualization of the report done in Tableau.
  - Also attached as a file for your convenience.


### Extra


- `mart_clickhouse.sql` – the ClickHouse SQL query used in the collab (**Code** section).
- `mart_redshift.sql` – the AWS Redshift SQL query from which the ClickHouse one was built.
- `sanity_redshift.sql` – the AWS Redshift SQL queries used for data sanity checks and a little EDA.



## Comments

### Sanity

All is good sanity-wise:

1. Only `fact_ads` has duplicates (on the `fact_ts, ad_id,
platform` key).
1. One user has **at most one** registration/installation.
1. One user **may have many** payments (from 0 to 4).
1. The ad impression day **does match** the user registration day, the assumption holds.
1. Payments happen only **after** registration (as they should).

### EDA & Data Issues

**Time from registration to payment** — nothing interesting (in this particular dataset), can neglect while building mart.

\

**Scale imbalance between costs and payments:**
- The best-performing advertisement **loses 117K units** (currency is unknown).
- Overall ROMI is around **-94%** (and so are all the ROMIs no matter the breakdown).
- Cost per Installation is **2137 units** while average revenue per user is **127 units** (average revenue per paying user is not much better at **213 units**).
  - ARPU and ARPPU are not really fair here (since users have not played that much yet), but even in the current state serve as a good illustration of my point.


Seeing this gave me flashbacks: the same thing happened when I was doing [a test task for the IntellectoKids company](https://github.com/ib-rain/test_tasks/blob/main/intellectokids/Orlov_task_1.ipynb) (a closer look reveals that the cost data is very similar). Back then I assumed that cost data was given in **cents** while the user payments data was in **dollars**. For this case I shall assume that costs are given in **"pseudo-Eurocents"** while user payments are in **"pseudo-Euros"**. I.e. in order to convert them to the common units I need to divide costs by **100**. After doing so (in the Tableau workbook):
- The best-performing advertisement **wins 9K units**.
- Overall ROMI is around **+500%** ("a good one" according to [the rule of thumb](https://dashthis.com/kpi-examples/return-on-marketing-investment/#:~:text=That%20said%2C%20the%20rule%20of,to%20the%20sector%20of%20activity.)).
- Cost per Installation is now around **21 units** which is a much better pair to the ARPU-ARPPU mentioned above.



### Mart

1.   Handling (potential) duplicates:
  - `fact_payment`: If there is a duplication on the `fact_ts, user_id` level, I assume it's a data bug since it's hard to buy two things at the exact same time so I take only the `max`.
  - `fact_ads`: If there is a duplication on `fact_ts, platform-ad_id` level, I assume there is a split since tracking impression is hard and there might be split entries so I `sum` them.
2. Losing fields:
  - `fact_reg.entry_point` does not provide an opportunity for telling interesting and actionable story (checked and visualized off-screen) and cannot be included into mart without risk of duplication — thus, it's removed from the analysis.
  - `fact_payment.fact_ts` is not really interesting either (checked in **EDA**), I decided to drop it by aggregating the payment on the `user_id` level: it makes the daily breakdown of the total revenue a bit misleading, but only a bit since the user is tied to the day of their registration/installation 1-to-1.
3. Primary key: mart is built on the `date, ad_id, platform` level.



### Visualization

#### Proud

1. **Story**: the dashboard tells a story — it shows how successful (or not) has the campaign been from overall KPIs above to the categorical and temporal breakdowns below.
2. **Colours**:
  - **Cost, Revenue, Impressions, Installations** and the **ROMI-Profit-Payers-Payment tuple** have consistent colours throughout the whole dashboard.
  - The **Impresions, Installations**, and **the tuple ones** form visually appealing coloured horizontal lines in the KPI row.
3. **Fractal-ity**: every tooltip is configured to fractally repeat the same story in the relevant breakdown (the order is retained).
4. **Patametrization**:
  - The **Partition** paramete allows an easy breakdown by **Platform, Date, Ad ID**.
  - The **Daily Installations** chart has a dynamic (and meaningful) title.
  - The **Ad ID** breakdown is complex: the **top N by Installation** set allows to collapse multiple **Ad IDs** into one chart entry.



#### Not Proud



1. **Complexity**: this viz is not that easy to use — a BI engineer might need an onboarding, a non-technical user will need training.
2. **Padding**: the **Total, Cost per, Funnel Stats** KPIs are aligned left to make numbers easier to read, but I find it a bit awkward.
3. **Compromise**: it's impossible to comprehensivly visualize the **Ad ID** level of **Profit/Cost/Revenue** table and **Daily Installations** time series at the same time since the best value of **N** for table is the worst configuration for the chart and vice versa. The hope is that an easily changeable parameter will allow user to seamlessly switch between the two.



## Code

In [8]:
# # example
# %%bash
# ./clickhouse local -q "
# SELECT * FROM file('fact_ads.csv', CSVWithNames)
# LIMIT 10
# FORMAT CSVWithNames
# " > example_result.csv

In [9]:
%%bash
./clickhouse local -q "with
    payments as (
        select
            toDate(fact_ts) as ts,
            user_id,

            max(amount)     as amount
        from file('fact_payment.csv', CSVWithNames)
        group by
            1, 2
    ),

    advertisements as (
        select
            toDate(fact_ts)  as ts,
            platform,
            ad_id,

            sum(cost)        as cost,
            sum(impressions) as impressions
        from file('fact_ads.csv', CSVWithNames)
        group by
            1, 2, 3
    ),

    installations as ( --Registrations = installations.
        select
            toDate(fact_ts) as ts,
            user_id,
            ad_id,
            platform

            --Unfortunately, an unnecessary field with uninteresting stats, dropping.
            --min(entry_point) as entry_point
        from file('fact_reg.csv', CSVWithNames)
        group by
            1, 2, 3, 4
    ),

    payments_agg as (
        select
            user_id,

            sum(amount) as amount
        from payments as pay
        group by
            1
    ),

    reg_pay_agg as (
        select
            toDate(ins.ts)         as date,
            ins.ad_id,
            ins.platform,

            uniqExact(ins.user_id) as number_of_installations_,
            uniqExact(pa.user_id)  as number_of_payers_,
            sum(pa.amount)         as payment_amount_
        from installations as ins
        left join payments_agg as pa
        on ins.user_id = pa.user_id
        group by
            1, 2, 3
    ),

    ads_agg as (
        select
            toDate(ts)       as date,
            ad_id,
            platform,

            sum(cost)        as cost_,
            sum(impressions) as impressions_
        from advertisements
        group by
            1, 2, 3
    ),

    final as (
        select
            ada.date,
            ada.ad_id,
            ada.platform,

            max(ada.cost_)                    as cost,
            max(ada.impressions_)             as impressions,

            max(rpa.number_of_installations_) as number_of_installations,
            max(rpa.number_of_payers_)        as number_of_payers,
            max(rpa.payment_amount_)          as payment_amount
        from ads_agg as ada
        left join reg_pay_agg as rpa
        on
            (ada.date, ada.ad_id, ada.platform) =
            (rpa.date, rpa.ad_id, rpa.platform)
        group by
            1, 2, 3
    )

select
    date,
    ad_id,
    platform,
    cost,
    impressions,
    number_of_installations,
    number_of_payers,
    payment_amount
from final
order by
    1, 2, 3
FORMAT CSVWithNames;
" > test_mart.csv